<a href="https://colab.research.google.com/github/gurudattamanpreet/Practice/blob/main/Titanic_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score,classification_report
from sklearn.preprocessing import StandardScaler,MinMaxScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline as Imbpipeline

In [2]:
df=pd.read_csv('https://raw.githubusercontent.com/gurudattamanpreet/datasets/refs/heads/main/Titanic.csv')

In [3]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
df.shape

(891, 12)

In [5]:
df['Survived'].value_counts()

,count
Survived,
0,549
1,342


In [6]:
df.drop(['PassengerId','Name','Ticket','Cabin'],axis=1,inplace=True)

In [7]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [8]:
df.isna().sum()

,0
Survived,0
Pclass,0
Sex,0
Age,177
SibSp,0
Parch,0
Fare,0
Embarked,2


In [9]:
df.shape

(891, 8)

In [10]:
df.duplicated().sum()

np.int64(111)

In [11]:
df=df.drop_duplicates()

In [12]:
df.shape

(780, 8)

In [13]:
X_train,X_test,y_train,y_test=train_test_split(df.drop(['Survived'],axis=1),df['Survived'],test_size=0.2,random_state=1,stratify=df['Survived'])

In [14]:
X_train.shape

(624, 7)

In [15]:
X_test.shape

(156, 7)

In [16]:
X_train.describe().T

,count,mean,std,min,25%,50%,75%,max
Pclass,624.0,2.235577,0.857757,1.00,1.00,3.0,3.000,3.0000
Age,541.0,30.004473,15.079462,0.42,20.00,29.0,39.000,80.0000
SibSp,624.0,0.540064,1.050108,0.00,0.00,0.0,1.000,8.0000
Parch,624.0,0.429487,0.861982,0.00,0.00,0.0,1.000,6.0000
Fare,624.0,35.383153,54.416871,0.00,8.05,16.0,34.375,512.3292


In [17]:

num_col=X_train.select_dtypes('number').columns.tolist()
cat_col=X_train.select_dtypes('object').columns.tolist()

num_pipeline=SkPipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())])
cat_pipeline=SkPipeline([('impute',SimpleImputer(strategy='most_frequent')),('encode',OneHotEncoder(drop='first',handle_unknown='ignore',sparse_output=False))])

preprocessor=ColumnTransformer([('num',num_pipeline,num_col),('cat',cat_pipeline,cat_col)],remainder='passthrough')

In [18]:
pipe=SkPipeline([
    ('preprocessor',preprocessor),
    ('model',Rand())])

In [19]:
pipe.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  ['Pclass', 'Age', 'SibSp',
                                                   'Parch', 'Fare']),
                                                 ('cat',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encode',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Sex', 'Embarked'])])),
                ('model', LogisticRegression())])

In [20]:
y_pred_train=pipe.predict(X_train)
y_pred_test=pipe.predict(X_test)

In [21]:
accuracy_score(y_test,y_pred_test)

0.75

In [22]:
accuracy_score(y_train,y_pred_train)

0.7932692307692307

In [23]:
print(classification_report(y_test,y_pred_test))

              precision    recall  f1-score   support

           0       0.78      0.80      0.79        92
           1       0.70      0.67      0.69        64

    accuracy                           0.75       156
   macro avg       0.74      0.74      0.74       156
weighted avg       0.75      0.75      0.75       156



In [25]:
print(classification_report(y_train,y_pred_train))

              precision    recall  f1-score   support

           0       0.82      0.83      0.82       366
           1       0.75      0.74      0.75       258

    accuracy                           0.79       624
   macro avg       0.79      0.79      0.79       624
weighted avg       0.79      0.79      0.79       624

